In [27]:
import polars as pl
from utils import getConnection
from utils import cleanContracts
from utils import readCSV

pl.Config.set_tbl_rows(100) # Show up to 100 rows

con, dataset_path = getConnection() # Create the duckDB connection
readCSV(con, dataset_path) # Read CSV and create temp contracts table
cleanContracts(con) # Clean the data before we begin our financial analysis

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

# Create temp table where one row represents the latest state of the contract

In [28]:
# This simplifies later queries 
con.sql("""CREATE TEMP TABLE contracts_latest AS
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY reference_number
               ORDER BY contract_date DESC
           ) AS rn
    FROM contracts_clean
) t
WHERE rn = 1;
""")

# Top 10 vendors by contract value

In [49]:
with pl.Config(float_precision=2, thousands_separator=','):
    print(con.sql("""
SELECT
    ROW_NUMBER() OVER (ORDER BY total_earned DESC) AS rank,
    vendor_name,
    total_contracts,
    total_earned
FROM (
SELECT
    vendor_name,
    COUNT(*) AS total_contracts,
    SUM(contract_value) AS total_earned
    FROM contracts_latest
    GROUP BY vendor_name
    ORDER BY total_earned DESC
    ) t
""").pl().limit(10))



shape: (10, 4)
┌──────┬─────────────────────────────────┬─────────────────┬───────────────────┐
│ rank ┆ vendor_name                     ┆ total_contracts ┆ total_earned      │
│ ---  ┆ ---                             ┆ ---             ┆ ---               │
│ i64  ┆ str                             ┆ i64             ┆ f64               │
╞══════╪═════════════════════════════════╪═════════════════╪═══════════════════╡
│ 1    ┆ Irving Shipbuilding Inc         ┆ 4               ┆ 20,402,436,551.91 │
│ 2    ┆ BANCTEC (CANADA), INC.          ┆ 8               ┆ 20,031,143,258.03 │
│ 3    ┆ General Dynamics Land Systems … ┆ 54              ┆ 8,625,542,848.69  │
│ 4    ┆ PCL CONSTRUCTORS CANADA INC.    ┆ 40              ┆ 8,314,979,097.55  │
│ 5    ┆ I.M.P Group Limited             ┆ 3               ┆ 8,166,089,765.70  │
│ 6    ┆ Irving Shipbuilding Inc.        ┆ 1               ┆ 8,010,267,025.00  │
│ 7    ┆ Sikorsky International Operati… ┆ 7               ┆ 7,607,385,308.00  │
│ 8    ┆ Casc

# Top 10 vendors by total contracts

In [ ]:
# This simplifies later queries 
con.sql(""" 
    SELECT
    ROW_NUMBER() OVER (ORDER BY total_contracts DESC) AS rank,
    vendor_name,
    total_contracts,
    ROUND(total_earned, 2) AS total_earned
FROM (
SELECT
    vendor_name,
    COUNT(*) AS total_contracts,
    SUM(contract_value) AS total_earned
    FROM contracts_latest
    GROUP BY vendor_name
    ORDER BY total_contracts DESC
    ) t
""").limit(10)

┌───────┬─────────────────────────────────────────┬─────────────────┬───────────────┐
│ rank  │               vendor_name               │ total_contracts │ total_earned  │
│ int64 │                 varchar                 │      int64      │    double     │
├───────┼─────────────────────────────────────────┼─────────────────┼───────────────┤
│     1 │ MCKESSON CANADA CORPORATION             │            2583 │  139153017.57 │
│     2 │ Veritaaq Technology House Inc.          │            1849 │  740771409.63 │
│     3 │ CANADIAN CORPS OF COMMISSIONAIRES       │            1761 │  416808632.58 │
│     4 │ IMPERIAL OIL                            │            1547 │   92228007.67 │
│     5 │ SIMEX DEFENCE INC. / DEFENSE SIMEX INC. │            1424 │  169541925.12 │
│     6 │ MCKESSON CANADA                         │            1304 │   70408808.66 │
│     7 │ Unisource Technology Inc                │            1052 │    90127419.9 │
│     8 │ SHELL                                   │   

# How many contracts have been amended? 

In [ ]:
# Over 143k contracts have been amended
con.sql(""" 
SELECT
    COUNT(*) AS contracts_with_amendments
FROM (
    SELECT
        reference_number,
        COUNT(*) AS version_count
    FROM
        contracts_clean
    GROUP BY
        reference_number
    HAVING
        COUNT(*) > 1   -- if the count is more than 1, it means there are amendments
) AS amended_contracts;
""")

┌───────────────────────────┐
│ contracts_with_amendments │
│           int64           │
├───────────────────────────┤
│                    143148 │
└───────────────────────────┘

# Amendment count vs. amendment impact

In [ ]:
con.sql("""
    SELECT
    reference_number,
    MAX(vendor_name) AS vendor_name, -- same for all versions
    ROUND(SUM(COALESCE(amendment_value, 0)), 2) AS total_amended,
    COUNT(*) - 1 AS num_amendments, -- # of amendments (exclude original)
    ROUND(AVG(COALESCE(amendment_value, 0)), 2) AS avg_amendment_value
FROM
    contracts_clean -- BACK TO FULL TABLE TO GET ALL VERSIONS
GROUP BY
    reference_number
HAVING
    COUNT(*) > 1 -- only amended contracts
ORDER BY
    num_amendments DESC;
""").limit(10)

┌──────────────────────┬──────────────────────────────┬───────────────┬────────────────┬─────────────────────┐
│   reference_number   │         vendor_name          │ total_amended │ num_amendments │ avg_amendment_value │
│       varchar        │           varchar            │    double     │     int64      │       double        │
├──────────────────────┼──────────────────────────────┼───────────────┼────────────────┼─────────────────────┤
│ C-2021-2022-Q4-00379 │ SveMy Construction Ltd.      │ 2914852202.18 │             18 │         153413273.8 │
│ C-2021-2022-Q2-01435 │ PCL CONSTRUCTORS CANADA INC. │ 2259999999.99 │              2 │        753333333.33 │
│ C-2024-2025-Q2-00053 │ WSP CANADA INC.              │ 1150731010.46 │             28 │         39680379.67 │
│ C-2020-2021-Q2-00398 │ gcproc                       │  1098914942.0 │             10 │         99901358.36 │
│ C-2024-2025-Q4-00001 │ XEROX CANADA LTD. - OTTAWA   │ 1034698681.85 │             72 │         14173954.55 │
│

# Top 10 contracts by amendment impact

In [88]:
con.sql("""
    -- Amendment impact using contracts_latest (one row per contract)
    SELECT
        reference_number,
        vendor_name,
        ROUND(contract_value - original_value, 2) AS amendment_impact
    FROM
        contracts_latest
    ORDER BY
        amendment_impact DESC
    LIMIT 10;
""")

┌───────────────────────┬──────────────────────────────────────┬──────────────────┐
│   reference_number    │             vendor_name              │ amendment_impact │
│        varchar        │               varchar                │      double      │
├───────────────────────┼──────────────────────────────────────┼──────────────────┤
│ C-2024-2025-Q2-02238  │ BANCTEC (CANADA), INC.               │   20012319127.56 │
│ C-2025-2026-Q1-04643  │ Irving Shipbuilding Inc              │     3516001276.9 │
│ C-2023-2024-Q4-005303 │ I.M.P Group Limited                  │    3292160898.11 │
│ C-2023-2024-Q4-005304 │ Irving Shipbuilding Inc              │     3200725032.5 │
│ C-2023-2024-Q1-04775  │ Irving Shipbuilding Inc              │    2577923398.34 │
│ C-2024-2025-Q1-04719  │ L-3 Communications MAS (Canada) Inc. │     2296174597.0 │
│ C-2022-2023-Q4-04984  │ I.M.P Group Limited                  │    2254550863.55 │
│ C-2022-2023-Q4-04981  │ L-3 Communications MAS (Canada) Inc. │     2195013